In [ ]:
import time
from datetime import datetime,timezone

import numpy as np
import zmq

ctx = zmq.Context.instance()

socket = ctx.socket(zmq.PUB)
socket.bind("tcp://127.0.0.1:5556")

price = 1.1
spread = 0.0005
rng = np.random.default_rng(seed=42)

while True:
    shock = rng.normal(scale = 0.0002)
    price *= 1.0 + shock
    ask = price + spread
    bid = price - spread
    volume = int(rng.integers(100000, 1000000000))
    now = datetime.now(timezone.utc).isoformat()
    msg = f"{now} EURUSD {bid:.5f} {ask:.5f} {volume}"
    socket.send_string(msg)
    time.sleep(0.5)
 

In [ ]:
import time
from datetime import datetime, timezone
import pandas as pd
import zmq

ctx = zmq.Context.instance()
socket = ctx.socket(zmq.PUB)
socket.bind("tcp://127.0.0.1:5556")

speed = 10.0  # 10x accelerated replay

# load historical data
df = pd.read_csv("eurusd_ticks.csv")

for i in range(len(df)):
    timestamp = df['timestamp'][i]
    bid       = df['bid'][i]
    ask       = df['ask'][i]
    volume    = df['volume'][i]
    
    msg = f"{timestamp} EURUSD {bid:.5f} {ask:.5f} {int(volume)}"
    socket.send_string(msg)
    time.sleep(0.5 / speed)

In [ ]:
import zmq
from collections import deque
import math

# connecting
ctx = zmq.Context.instance()

socket = ctx.socket(zmq.SUB)
socket.connect("tcp://127.0.0.1:5556")

socket.setsockopt_string(zmq.SUBSCRIBE, "EURUSD")

# rolling return
class OnlineMomentum:
    def __init__(self,window:int = 10) -> None:
        self.window = window
        self.buffer = deque(maxlen = window)
        self.sum_ret = 0.0
    def update(self, r_t: float) -> float:
        if len(self.buffer) == self.window:
            oldest = self.buffer[0]
            self.sum_ret -= oldest
        self.buffer.append(r_t)
        self.sum_ret += r_t
        if len(self.buffer) < self.window:
            return 0.0
        return self.sum_ret / float(self.window)

mom = OnlineMomentum(window = 20)
prev_mid = None


mom  = OnlineMomentum(window=20)  # rolling mean of returns
spread_tracker = OnlineMomentum(window=20)  # rolling mean of spread
vol_tracker    = OnlineMomentum(window=20)  # rolling mean of r_t**2

# get messege

while True:
    msg = socket.recv_string()
    parts = msg.split()
    bid = float(parts[2])
    ask = float(parts[3])
    mid = (bid + ask) / 2
    spread = ask - bid
    mean_spread = spread_tracker.update(spread)

    if prev_mid is not None:
        # compute simple return
        r_t = (mid - prev_mid) / prev_mid

        # momentum signal
        m_t = mom.update(r_t)
        signal_t = 1.0 if m_t > 0.0 else -1.0 if m_t < 0.0 else 0.0

        # volatility = sqrt of rolling mean of squared returns
        vol = math.sqrt(vol_tracker.update(r_t**2))

        print(f"mid={mid:.5f} | "
              f"mean_spread={mean_spread:.5f} | "
              f"vol={vol:.6f} | "
              f"signal={signal_t}")

    prev_mid = mid 


In [ ]:
import zmq

# subscriber
ctx    = zmq.Context.instance()
socket = ctx.socket(zmq.SUB)
socket.connect("tcp://127.0.0.1:5556")
socket.setsockopt_string(zmq.SUBSCRIBE, "EURUSD")

# set up a csv file
f = open("ticks.csv", "w", newline="")
writer = csv.writer(f)
writer.writerow(["timestamp", "symbol", "bid", "ask", "volume"])

# get message and write in the csv file ( collecting the ticks)
counter = 0
while True:
    msg       = socket.recv_string()  
    parts     = msg.split()             
    timestamp = parts[0]
    symbol    = parts[1]
    bid       = float(parts[2])
    ask       = float(parts[3])
    volume    = int(parts[4])

    writer.writerow([timestamp, symbol, bid, ask, volume])
    f.flush()   # write immediately to disk - AI suggested
    counter += 1
    print(f"stored tick {counter}: {timestamp} {bid:.5f} {ask:.5f}")

df = pd.read_csv("ticks.csv")

for i in range(1, len(df)):
    prev_ts = df["timestamp"][i-1]
    curr_ts = df["timestamp"][i]
    if curr_ts < prev_ts:
        print(f"ORDER VIOLATION at row {i}")
    else:
        print(f"row {i} ok")

In [ ]:
import zmq
import math
from datetime import datetime, timezone
from collections import deque

ctx = zmq.Context.instance()
socket = ctx.socket(zmq.SUB)
socket.connect("tcp://127.0.0.1:5556")
socket.setsockopt_string(zmq.SUBSCRIBE, "EURUSD")

latencies = []

While True:
    msg = socket.recv_string()
    recv_time = datetime.now(timezone.utc)

    parts = msg.split()
    send_time = datetime.fromisoformat(parts[0])

    latency_ms = (recv_time - send_time).total_seconds() * 1000
    latencies.append(latency_ms)

    if len(latencies) >=20:
        mean_l = sum(latencies) / len(latencies)
        var_l  = sum((x - mean_l)**2 for x in latencies) / len(latencies)
        std_l  = math.sqrt(var_l)
        max_l  = max(latencies)
        min_l  = min(latencies)
        print(f"\n--- latency summary (last 20 ticks) ---")
        print(f"mean : {mean_l:.3f} ms")
        print(f"std  : {std_l:.3f} ms")
        print(f"max  : {max_l:.3f} ms")
        print(f"min  : {min_l:.3f} ms")

In [1]:
import math

def update_mean_var(n, mean, m2, x):
    n_new    = n + 1
    delta    = x - mean                    
    mean_new = mean + delta / n_new        
    delta2   = x - mean_new               
    m2_new   = m2 + delta * delta2       
    return n_new, mean_new, m2_new

# test on short return sequence
returns = [0.01, -0.02, 0.03, -0.01, 0.02]

n    = 0
mean = 0.0
m2   = 0.0

for x in returns:
    n, mean, m2 = update_mean_var(n, mean, m2, x)
    var = m2 / n if n > 0 else 0.0
    std = math.sqrt(var)

In [2]:
import math
from collections import deque

# Rolling Window
class RollingVolatility:
    def __init__(self, window=20):
        self.window  = window
        self.buffer  = deque(maxlen=window)
        self.sum_sq  = 0.0          

    def update(self, r_t):
        if len(self.buffer) == self.window:
            oldest      = self.buffer[0]
            self.sum_sq -= oldest       # kick oldest value
        r_sq = r_t ** 2
        self.buffer.append(r_sq)
        self.sum_sq += r_sq
        if len(self.buffer) < self.window:
            return 0.0
        return math.sqrt(self.sum_sq / self.window)  # rolling vol


#  EWMA Volatility: assigna higher weight to the latest data
class EWMAVolatility:
    def __init__(self, lam=0.8):
        self.lam   = lam
        self.ewma  = 0.0               

    def update(self, r_t):
        self.ewma = self.lam * self.ewma + (1 - self.lam) * r_t**2
        return math.sqrt(self.ewma)    # volatility


#  test on short return sequence
returns = [0.01, -0.02, 0.03, -0.01, 0.02,
           0.01,  0.04, -0.03, 0.01, -0.02,
           0.05, -0.01,  0.02, 0.03, -0.04,
           0.01, -0.02,  0.01, 0.02,  0.03,
           0.10,  0.08,  0.09, 0.10, -0.09] 

roll = RollingVolatility(window=20)
ewma = EWMAVolatility(lam=0.8)

print("step | r_t    | rolling_vol | ewma_vol")
print("-----|--------|-------------|----------")
for i, r_t in enumerate(returns):
    r_vol  = roll.update(r_t)
    e_vol  = ewma.update(r_t)
    print(f"{i+1:4d} | {r_t:6.3f} | {r_vol:.6f}  | {e_vol:.6f}")


step | r_t    | rolling_vol | ewma_vol
-----|--------|-------------|----------
   1 |  0.010 | 0.000000  | 0.004472
   2 | -0.020 | 0.000000  | 0.009798
   3 |  0.030 | 0.000000  | 0.016025
   4 | -0.010 | 0.000000  | 0.015015
   5 |  0.020 | 0.000000  | 0.016135
   6 |  0.010 | 0.000000  | 0.015109
   7 |  0.040 | 0.000000  | 0.022419
   8 | -0.030 | 0.000000  | 0.024127
   9 |  0.010 | 0.000000  | 0.022038
  10 | -0.020 | 0.000000  | 0.021646
  11 |  0.050 | 0.000000  | 0.029578
  12 | -0.010 | 0.000000  | 0.026830
  13 |  0.020 | 0.000000  | 0.025610
  14 |  0.030 | 0.000000  | 0.026546
  15 | -0.040 | 0.000000  | 0.029728
  16 |  0.010 | 0.000000  | 0.026963
  17 | -0.020 | 0.000000  | 0.025722
  18 |  0.010 | 0.000000  | 0.023437
  19 |  0.020 | 0.000000  | 0.022791
  20 |  0.030 | 0.024900  | 0.024404
  21 |  0.100 | 0.033392  | 0.049764
  22 |  0.080 | 0.037616  | 0.057106
  23 |  0.090 | 0.042131  | 0.065030
  24 |  0.100 | 0.047645  | 0.073370
  25 | -0.090 | 0.051527  | 0.076

In [6]:
import math

class OnlineZScore:
    def __init__(self, warmup =20):
        self.n = 0
        self.mean=0
        self.warmup = warmup
        self.m2 = 0

    def update(self, x):
        self.n +=1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.m2 += delta * delta2

        if self.n < self.warmup:
            return 0.0

        std = math.sqrt(self.m2/self.n)

        return (x-self.mean)/std

returns = [0.01, -0.02, 0.03, -0.01, 0.02,
           0.01,  0.04, -0.03, 0.01, -0.02,
           0.05, -0.01,  0.02, 0.03, -0.04,
           0.01, -0.02,  0.01, 0.02,  0.03,
           0.10,  0.08,  0.09, 0.10, -0.09] 

zscore = OnlineZScore(warmup=20)

for i, r_t in enumerate(returns):
    z = zscore.update(r_t)

    if z > 1.0 :
        signal = 1.0
    elif z < -1.0:
        signal = -1.0
    else:
        signal = 0.0

    std = math.sqrt(zscore.m2 / zscore.n) if zscore.n > 1 else 0.0

In [ ]:
import math
import random
import zmq

class EpsilonGreedyBandit:
    def __init__(self, epsilon=0.1):
        self.epsilon = epsilon   

        # each arm: count and mean reward
        self.counts  = [0, 0]       
        self.means   = [0.0, 0.0]   

    def select_arm(self):
        # explore: random arm
        # 10% this will hapoens and choose randomly
        if random.random() < self.epsilon:
            return random.randint(0, 1)
        # 90% this will happens and exploit best arm so far
        return 0 if self.means[0] >= self.means[1] else 1

    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        # mean_{t+1} = mean_t + (reward - mean_t) / n
        self.means[arm] += (reward - self.means[arm]) / n


# arm 0: momentum strategy  (assume true mean return = 0.010)
# arm 1: mean reversion      (assume true mean return = 0.015)

ctx    = zmq.Context.instance()
socket = ctx.socket(zmq.SUB)
socket.connect("tcp://127.0.0.1:5556")
socket.setsockopt_string(zmq.SUBSCRIBE, "EURUSD")

bandit            = EpsilonGreedyBandit(epsilon=0.1)  # defined before loop
cumulative_regret = 0.0
prev_mid          = None
step              = 0

while True:
    msg    = socket.recv_string()
    parts  = msg.split()
    bid    = float(parts[2])
    ask    = float(parts[3])
    mid    = (bid + ask) / 2

    if prev_mid is not None:
        r_t = (mid - prev_mid) / prev_mid 

        arm    = bandit.select_arm()

        # arm0: momentum signal
        if arm == 0:
            signal = 1.0 if r_t > 0 else -1.0
        # arm1: mean reversion signal
        else:
            signal = -1.0 if r_t > 0 else 1.0

        # reward = signal × next return 
        reward = signal * r_t
        bandit.update(arm, reward)

        best_mean = max(bandit.means)  
        regret  = best_mean - reward
        cumulative_regret += regret
        step  += 1
        
    prev_mid = mid
 

In [ ]:
##the batch calculating the statistics using the historical data all at once
## the online method use new tick to generate / calculate like a loop.

## theritical the statistics of using those two method should be the same
 # but when the data is large, offline method can be slow because calculating too much data at the same time. but online one use mean_t adding increase, so has like only 2 data at a time.
